# ️ Glu-Stock: 03_EXECUTION_ENGINE
**Phase**: Risk Management & Trade Execution

This notebook retrieves signals from the Firebase `signals` queue, calculates position sizes (ATR/Kelly), and updates the global `trades` state in the cloud.

In [ ]:
#  SECTION 1: INSTALLATION
!pip install -q yfinance firebase-admin pandas python-dotenv


In [ ]:
# [INFO] SECTION 2: INFRASTRUCTURE (Firebase, Secrets & Universe)
import json, os, firebase_admin, joblib, numpy as np, pandas as pd, yfinance as yf, warnings
from firebase_admin import credentials, firestore
from datetime import datetime
warnings.filterwarnings('ignore')

try:
    from kaggle_secrets import UserSecretsClient
    IS_KAGGLE = True
except ImportError:
    IS_KAGGLE = False

class KaggleInfra:
    @staticmethod
    def load_secrets():
        if IS_KAGGLE:
            user_secrets = UserSecretsClient()
            try:
                raw = user_secrets.get_secret("FIREBASE_KEY_JSON")
                return {"key": json.loads(raw)}
            except Exception as e:
                print(f"[ERROR] FIREBASE_KEY_JSON missing or invalid! Error: {e}")
                return {"key": None}
        else:
            from dotenv import load_dotenv
            load_dotenv()
            raw = os.getenv("FIREBASE_KEY_JSON")
            if not raw: return {"key": None}
            return {"key": json.loads(raw)}

class FirebaseHandler:
    def __init__(self, secrets):
        if not firebase_admin._apps:
            if not secrets.get('key'):
                raise ValueError("FIREBASE_KEY_JSON is missing. Check Kaggle Secrets.")
            cred = credentials.Certificate(secrets['key'])
            firebase_admin.initialize_app(cred)
        self.db = firestore.client()
    def get_and_clear_queue(self, queue_name: str):
        docs = self.db.collection(f"glu_stock_queue_{queue_name}").get()
        tasks = []
        for doc in docs:
            dt = doc.to_dict()
            tasks.append(dt.get('payload', dt))
            doc.reference.delete()
        return tasks
    def push_task(self, queue_name: str, data):
        self.db.collection(f"glu_stock_queue_{queue_name}").add({'payload': data, 'timestamp': datetime.now().isoformat()})
    def log_event(self, phase, details):
        self.db.collection("glu_stock_history").add({'timestamp': datetime.now().isoformat(), 'phase': phase.upper(), 'details': details})


In [ ]:
#  SECTION 3: CORE LOGIC (Risk & Execution)
class RiskManager:
    def calculate_size(self, price, conviction, total_equity=100000000):
        # Simple ATR-like sizing demo
        return int((total_equity * 0.02 * conviction) / price)

class TradingAgent:
    def __init__(self, fb):
        self.fb = fb
        self.risk = RiskManager()

    def execute_signals(self, signals):
        for ticker, data in signals.items():
            print(f" Executing trade for {ticker}...")
            shares = self.risk.calculate_size(data['price'], data['conviction'])
            if shares > 0:
                trade_data = {
                    'ticker': ticker,
                    'shares': shares,
                    'entry_price': data['price'],
                    'entry_date': datetime.now().isoformat(),
                    'conviction': data['conviction'],
                    'status': 'OPEN'
                }
                self.fb.insert_trade(trade_data)
                self.fb.log_event("EXECUTION", f"Opened {ticker} @ {data['price']} ({shares} shares)")

In [ ]:
#  SECTION 4: MAIN EXECUTION
def run_execution():
    secrets = KaggleInfra.load_secrets()
    fb = FirebaseHandler(secrets)
    
    # 1. Pull signals queue
    signal_batches = fb.get_and_clear_queue("signals")
    if not signal_batches: print(" Queue empty."); return
    
    all_signals = {}
    for batch in signal_batches: all_signals.update(batch)

    # 2. Execute
    agent = TradingAgent(fb)
    agent.execute_signals(all_signals)
    print(" Execution complete.")

run_execution()